# Setup

In [1]:
%pip install -q spuco==2.0.3 "matplotlib>=3.7.1,<3.11,!=3.9.*"

In [2]:
import torch
from spuco.utils import set_seed


device = torch.device("cuda")
set_seed(0)

# Dataset Creation

**Changing settings (Parts 3 and 4):** edit `spurious_correlation_strength` or `difficulty` in the cell below. After every change, rerun **this cell and all cells below it, in order**. This cell resets the random seed to 0, so each setting is trained from scratch under the same seed. Do not rerun only the training or evaluation cell.

In [13]:
from spuco.robust_train import ERM
from spuco.datasets import SpuCoMNIST, SpuriousFeatureDifficulty
import torchvision.transforms as T
from spuco.utils import set_seed

# Reset the seed so every rerun from this cell starts from the same state.
set_seed(0)

classes = [[0, 1], [2, 3], [4, 5], [6, 7], [8, 9]]
difficulty = SpuriousFeatureDifficulty.MAGNITUDE_SMALL

trainset = SpuCoMNIST(
    root="/data/mnist/",
    spurious_feature_difficulty=difficulty,
    spurious_correlation_strength=0.995,
    classes=classes,
    split="train",
    label_noise=0.001
)
trainset.initialize()

valset = SpuCoMNIST(
    root="/data/mnist/",
    spurious_feature_difficulty=difficulty,
    classes=classes,
    split="val"
)
valset.initialize()

testset = SpuCoMNIST(
    root="/data/mnist/",
    spurious_feature_difficulty=difficulty,
    classes=classes,
    split="test"
)
testset.initialize()

# Show a training example
T.ToPILImage()(trainset[1000][0]).resize((28,28))


In [14]:
trainset

In [15]:
# Show a training example
print(trainset.group_weights[(2, 2)])
print(trainset.group_weights[(2, 3)])
print((trainset[999][1]))
T.ToPILImage()(trainset[999][0]).resize((28,28))

0.18656778601783186
0.0002499791684026331
3


# Train ERM

In [16]:
from spuco.models import model_factory
from torch.optim import SGD


model = model_factory("lenet", trainset[0][0].shape, trainset.num_classes).to(device)
erm = ERM(
    model=model,
    num_epochs=3,
    trainset=trainset,
    batch_size=64,
    optimizer=SGD(model.parameters(), lr=1e-2, momentum=0.9, nesterov=True),
    device=device,
    verbose=False # set to True to see the progress
)

erm.train()

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:431: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:437: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


# Evaluate ERM

In [17]:
import pprint
from spuco.evaluate import Evaluator

evaluator = Evaluator(
    testset=testset,
    group_partition=testset.group_partition,
    group_weights=trainset.group_weights,
    batch_size=64,
    model=model,
    device=device,
    verbose=False # set to True to see the progress
)

evaluator.evaluate()

print()
pprint.pprint(evaluator.accuracies, indent=4, width=80)
print("\nWorst group accuracy: ", evaluator.worst_group_accuracy)
print("\nAverage accuracy:", evaluator.average_accuracy)

Evaluating group-wise accuracy: 100%|██████████| 25/25 [00:14<00:00,  1.67it/s]


{   (0, 0): 98.81796690307328,
    (0, 1): 100.0,
    (0, 2): 99.29078014184397,
    (0, 3): 99.52718676122932,
    (0, 4): 99.29078014184397,
    (1, 0): 99.26650366748166,
    (1, 1): 98.77750611246944,
    (1, 2): 99.50980392156863,
    (1, 3): 98.52941176470588,
    (1, 4): 98.52941176470588,
    (2, 0): 96.0,
    (2, 1): 97.06666666666666,
    (2, 2): 97.33333333333333,
    (2, 3): 98.66666666666667,
    (2, 4): 98.39572192513369,
    (3, 0): 98.49246231155779,
    (3, 1): 97.73299748110831,
    (3, 2): 98.2367758186398,
    (3, 3): 98.48866498740554,
    (3, 4): 97.22921914357683,
    (4, 0): 97.48110831234257,
    (4, 1): 97.22921914357683,
    (4, 2): 97.98488664987406,
    (4, 3): 98.23232323232324,
    (4, 4): 97.97979797979798}

Worst group accuracy:  ((2, 0), 96.0)

Average accuracy: 98.29960399029054


MAGNITUDE LARGE for SpuCo feature (EASY):

(2,2) Accuracy @ 0.995 SpuCo Strength: 100.00

Worst group accuracy:  ((0, 1), 0.0)

Average accuracy: 99.39380051662361

(2,3) Accuracy @ 0.995 SpuCo Strength: 0.0

(2,3) Accuracy @ 0.9 SpuCo Strength: 76.8

(2,3) Accuracy @ 0.965 SpuCo Strength: 34.4

---

MAGNITUDE MEDIUM for SpuCo feature (MEDIUM):

(2,3) Accuracy @ 0.995 SpuCo Strength: 93.06666

Worst group accuracy:  ((4, 1), 15.365239294710328)

Average accuracy: 99.63079004134254

---

MAGNITUDE SMALL for SpuCo feature (HARD):

(2,3) Accuracy @ 0.995 SpuCo Strength: 93.06666

Worst group accuracy:  ((2, 0), 96.0)

Average accuracy: 98.29960399029054